In [2]:
import numpy as np
import pandas as pd
from scipy.stats import norm

np.random.seed(42)

# Homework parameters
ALPHA = 0.95
K = 10
M = 50_000
LAMBDA = 0.97
THETA = 0.97
PORTFOLIO_VALUE = 1_000_000

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("All required libraries loaded!")


NumPy: 2.4.6
Pandas: 3.0.6
All required libraries loaded!


In [3]:
# %pip install yfinance

import yfinance as yf

tickers = ["MSFT", "AAPL"]

prices = yf.download(
    tickers,
    start="2011-08-31",
    end="2016-09-02",
    auto_adjust=True,
    progress=False
)["Close"]

prices = prices[tickers].dropna()

# Daily log returns
log_returns = np.log(prices / prices.shift(1)).dropna()

# Restrict sample to homework dates
log_returns = log_returns.loc[
    "2011-09-01":"2016-09-01"
]

print("Data shape:", log_returns.shape)
print("First date:", log_returns.index.min())
print("Last date:", log_returns.index.max())

display(log_returns.head())


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


Data shape: (1259, 2)
First date: 2011-09-01 00:00:00
Last date: 2016-09-01 00:00:00


Ticker,MSFT,AAPL
Date,,
2011-09-01,-0.014770,-0.009924
2011-09-02,-0.015767,-0.018489
2011-09-06,-0.011304,0.015098
2011-09-07,0.019026,0.010973
2011-09-08,0.008425,0.000547


In [4]:
market_caps = np.array([448.77, 575.11])

weights = market_caps / market_caps.sum()

# Dollar positions
positions = PORTFOLIO_VALUE * weights

print("MSFT weight:", weights[0])
print("AAPL weight:", weights[1])
print("MSFT position:", positions[0])
print("AAPL position:", positions[1])
print("Total:", positions.sum())


MSFT weight: 0.4383033167949369
AAPL weight: 0.5616966832050632
MSFT position: 438303.3167949369
AAPL position: 561696.6832050631
Total: 1000000.0


In [5]:

def ewma_update(mu, cov, x, lam=0.97, theta=0.97):
    """
    Update EWMA mean and covariance.
    mu: (2,)
    cov: (2, 2)
    x: (2,)
    """
    residual = x - mu

    mu_new = lam * mu + (1 - lam) * x

    cov_new = (
        theta * cov
        + (1 - theta) * np.outer(residual, residual)
    )

    return mu_new, cov_new


# Initial values
mu = np.zeros(2)
cov = np.zeros((2, 2))

# Iterate through historical log returns
for x in log_returns.to_numpy():
    mu, cov = ewma_update(
        mu, cov, x, LAMBDA, THETA
    )

sigma = np.sqrt(np.diag(cov))
rho = cov[0, 1] / (sigma[0] * sigma[1])

print("EWMA Mean:")
print(mu)

print("\nEWMA Covariance:")
print(cov)

print("\nDaily Volatility:")
print(sigma)

print("\nCorrelation:", rho)


EWMA Mean:
[0.00140453 0.00116554]

EWMA Covariance:
[[1.26271384e-04 3.32410178e-05]
 [3.32410178e-05 1.52044656e-04]]

Daily Volatility:
[0.01123705 0.01233064]

Correlation: 0.23990332176453716


In [6]:
z_alpha = norm.ppf(ALPHA)

# Portfolio loss mean and standard deviation
loss_mean = -positions @ mu
loss_std = np.sqrt(positions @ cov @ positions)

# Baseline 1-day VaR
var_1day = loss_mean + z_alpha * loss_std

# Square-root-of-time approximation
var_10day = np.sqrt(K) * var_1day

# Regulatory capital
reg_capital = 3 * var_10day

print(f"1-Day VaR (95%): ${var_1day:,.2f}")
print(f"10-Day VaR (95%): ${var_10day:,.2f}")
print(f"Regulatory Capital: ${reg_capital:,.2f}")


1-Day VaR (95%): $14,211.97
10-Day VaR (95%): $44,942.19
Regulatory Capital: $134,826.57


In [7]:
# AAPL = index 1
# MSFT = index 0

# Apple: unconditional -5 sigma shock
x_aapl = mu[1] - 5 * sigma[1]

# Microsoft conditional on Apple shock
mu_msft_cond = (
    mu[0]
    + rho * sigma[0] / sigma[1]
    * (x_aapl - mu[1])
)

sigma_msft_cond = (
    sigma[0] * np.sqrt(1 - rho**2)
)

# Microsoft: conditional -5 sigma shock
x_msft = mu_msft_cond - 5 * sigma_msft_cond

# Day 1 shock vector
shock = np.array([x_msft, x_aapl])

print(f"AAPL shock: {x_aapl:.4%}")
print(f"MSFT conditional mean: {mu_msft_cond:.4%}")
print(f"MSFT conditional std: {sigma_msft_cond:.4%}")
print(f"MSFT shock: {x_msft:.4%}")

print("\nDay 1 Shock Vector:")
print(shock)

# Day 1 linearized loss
day1_loss = -positions @ shock
print(f"\nDay 1 Loss: ${day1_loss:,.2f}")


AAPL shock: -6.0488%
MSFT conditional mean: -1.2075%
MSFT conditional std: 1.0909%
MSFT shock: -6.6619%

Day 1 Shock Vector:
[-0.06661898 -0.06048766]

Day 1 Loss: $63,175.04


In [8]:
rng = np.random.default_rng(42)

# 50,000 independent simulation paths
mu_paths = np.repeat(mu[None, :], M, axis=0)
cov_paths = np.repeat(cov[None, :, :], M, axis=0)

# Accumulated 10-day log returns
cumulative_returns = np.zeros((M, 2))

# ----- DAY 1: deterministic shock -----
x_day1 = np.broadcast_to(shock, (M, 2)).copy()

cumulative_returns += x_day1

# Update EWMA from Day 1 shock
residual = x_day1 - mu_paths

cov_paths = (
    THETA * cov_paths
    + (1 - THETA)
    * residual[:, :, None]
    * residual[:, None, :]
)

mu_paths = (
    LAMBDA * mu_paths
    + (1 - LAMBDA) * x_day1
)

# ----- DAYS 2 TO 10 -----
for day in range(2, K + 1):

    # Generate correlated normal returns
    z1 = rng.standard_normal(M)
    z2 = rng.standard_normal(M)

    var1 = np.maximum(cov_paths[:, 0, 0], 1e-15)
    cov12 = cov_paths[:, 0, 1]
    var2 = np.maximum(cov_paths[:, 1, 1], 1e-15)

    # Conditional normal sampling
    std1 = np.sqrt(var1)

    cond_mean2 = mu_paths[:, 1] + (
        cov12 / var1
    ) * (std1 * z1)

    cond_var2 = np.maximum(
        var2 - cov12**2 / var1, 0
    )

    x1 = mu_paths[:, 0] + std1 * z1
    x2 = cond_mean2 + np.sqrt(cond_var2) * z2

    x = np.column_stack([x1, x2])

    # Accumulate returns
    cumulative_returns += x

    # Update EWMA except after final day
    if day < K:
        residual = x - mu_paths

        cov_paths = (
            THETA * cov_paths
            + (1 - THETA)
            * residual[:, :, None]
            * residual[:, None, :]
        )

        mu_paths = (
            LAMBDA * mu_paths
            + (1 - LAMBDA) * x
        )

# Compute 10-day linearized portfolio losses
losses = -cumulative_returns @ positions

print("Number of simulated losses:", len(losses))
print("First five losses:")
print(losses[:5])


Number of simulated losses: 50000
First five losses:
[ 3889.97419183 53156.22937469 68209.61140467 36346.92111963
 42357.78390231]


In [9]:
# (a) Average 10-day portfolio loss
avg_loss = np.mean(losses)

# (b) Stressed 95% 10-day VaR
stressed_var = np.quantile(losses, ALPHA)

# (c) Frequency exceeding initial VaR
exceed_var = 100 * np.mean(losses > var_10day)

# (d) Frequency exceeding regulatory capital
exceed_capital = 100 * np.mean(losses > reg_capital)

results = pd.DataFrame({
    "Metric": [
        "(a) Average 10-Day Loss",
        "(b) Stressed 10-Day VaR (95%)",
        "(c) Exceed Initial 10-Day VaR",
        "(d) Exceed Regulatory Capital"
    ],
    "Value": [
        f"${avg_loss:,.2f}",
        f"${stressed_var:,.2f}",
        f"{exceed_var:.3f}%",
        f"{exceed_capital:.3f}%"
    ]
})

display(results)


,Metric,Value
0,(a) Average 10-Day Loss,"$69,327.98"
1,(b) Stressed 10-Day VaR (95%),"$150,030.33"
2,(c) Exceed Initial 10-Day VaR,69.360%
3,(d) Exceed Regulatory Capital,8.776%


## Results and Interpretation

Using 50,000 Monte Carlo simulation paths over a 10-day horizon, the estimated stressed portfolio risk measures are:

- **(a) Average 10-day loss:** USD 69,327.98
- **(b) Stressed 95% 10-day VaR:** USD 150,030.33
- **(c) Frequency exceeding initial 10-day VaR:** 69.360%
- **(d) Frequency exceeding regulatory capital:** 8.776%

The initial 10-day VaR is USD 44,942.19, and the corresponding regulatory capital requirement is USD 134,826.57.

After imposing a negative five-standard-deviation shock on Apple and a negative five-conditional-standard-deviation shock on Microsoft, the stressed 10-day VaR increases to approximately **3.34 times the initial VaR**.

The initial VaR exceedance frequency of 69.360% suggests that the pre-shock VaR substantially understates potential losses under the specified stress scenario. Furthermore, 8.776% of simulated losses exceed the regulatory capital threshold.

These results demonstrate how extreme market shocks, together with dynamically updated EWMA means and covariances, can significantly increase portfolio risk beyond its initial estimates.